In [16]:
# Import Libraries
import rasterio
import numpy as np
import os
import json
from datetime import datetime
print("Libraries imported successfully")

Libraries imported successfully


In [17]:
CONFIG = {
    "raw_path_past": "../data/raw/past_image.tif",
    "raw_path_recent": "../data/raw/recent_image.tif",
    "output_folder": "../data/processed"
}

In [18]:
os.makedirs(CONFIG["output_folder"], exist_ok=True)
print("Processed folder ready")

Processed folder ready


In [25]:
def compute_ndwi(input_path, output_path):
    
    with rasterio.open(input_path) as src:
        green = src.read(1).astype(float)
        nir = src.read(2).astype(float)
        profile = src.profile
    
    # Avoid division by zero
    denominator = (green + nir)
    denominator[denominator == 0] = np.nan
    
    ndwi = (green - nir) / denominator
    
    # Validate NDWI range
    ndwi = np.clip(ndwi, -1, 1)
    
    profile.update(dtype=rasterio.float32, count=1)
    
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(ndwi.astype(rasterio.float32), 1)
    
    return ndwi

In [26]:
past_ndwi = compute_ndwi(
    CONFIG["raw_path_past"],
    f"{CONFIG['output_folder']}/past_ndwi.tif"
)

print("Past NDWI computed and saved")

Past NDWI computed and saved


In [27]:
def print_stats(ndwi, label):
    print(f"{label} NDWI Stats:")
    print("  Min:", round(np.nanmin(ndwi), 3))
    print("  Max:", round(np.nanmax(ndwi), 3))
    print("  Mean:", round(np.nanmean(ndwi), 3))
    print("")

print_stats(past_ndwi, "Past")
print_stats(recent_ndwi, "Recent")

Past NDWI Stats:
  Min: -0.748
  Max: 0.341
  Mean: -0.132

Recent NDWI Stats:
  Min: -0.792
  Max: 0.387
  Mean: -0.123



In [28]:
metadata = {
    "processing": "NDWI calculation",
    "formula": "(Green - NIR) / (Green + NIR)",
    "band_order": {
        "band_1": "Green (B3)",
        "band_2": "NIR (B8)"
    },
    "past_ndwi_min": float(np.nanmin(past_ndwi)),
    "past_ndwi_max": float(np.nanmax(past_ndwi)),
    "recent_ndwi_min": float(np.nanmin(recent_ndwi)),
    "recent_ndwi_max": float(np.nanmax(recent_ndwi)),
    "timestamp": datetime.utcnow().isoformat()
}

with open("../output/preprocessing_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Preprocessing metadata saved")

Preprocessing metadata saved
